# 03 — Fine-tune with LLaMA-Factory (Hands-on!)

In this notebook we fine-tune **Qwen2.5-0.5B-Instruct** on our tiny `hello_alpaca` dataset using **LLaMA-Factory** with LoRA.

> 💡 This runs on a **free Google Colab T4 GPU** if you switch `finetuning_type` to QLoRA (`quantization_bit: 4`).

In [ ]:
!nvidia-smi

## 1. Install LLaMA-Factory

In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,metrics]" -q

## 2. Load our dataset

In [ ]:
import shutil
shutil.copy('../data/hello_alpaca.json', 'data/hello_alpaca.json')
shutil.copy('../data/dataset_info.json', 'data/dataset_info.json')
!cat data/dataset_info.json

## 3. Create the training config

This is the same as `configs/hello_lora.yaml` in the repo. The key lines:
- `stage: sft` → supervised fine-tuning
- `finetuning_type: lora` → we're training adapters, not full weights
- `lora_target: all` → apply LoRA to every linear layer
- `template: qwen` → the chat template matching our model

In [ ]:
config = """
### Model
model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
trust_remote_code: true

### Method
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 8
lora_alpha: 16

### Dataset
dataset: hello_alpaca
template: qwen
cutoff_len: 1024
overwrite_cache: true

### Output
output_dir: saves/hello-lora
logging_steps: 5
plot_loss: true

### Train
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
learning_rate: 5.0e-5
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
"""

with open('hello_lora.yaml', 'w') as f:
    f.write(config)

## 4. Train! 🚀

In [ ]:
!llamafactory-cli train hello_lora.yaml

## 5. Chat with your fine-tuned model

In [ ]:
chat_config = """
model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
adapter_name_or_path: saves/hello-lora
template: qwen
finetuning_type: lora
"""

with open('hello_chat.yaml', 'w') as f:
    f.write(chat_config)

In [ ]:
# Launch the interactive chat web UI
!llamafactory-cli webchat hello_chat.yaml

## 6. (Optional) Merge and export

To get a standalone model without the adapter:

In [ ]:
export_config = """
model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
adapter_name_or_path: saves/hello-lora
template: qwen
finetuning_type: lora
export_dir: models/hello-merged
"""

with open('hello_export.yaml', 'w') as f:
    f.write(export_config)

!llamafactory-cli export hello_export.yaml

## What's next?

- 🗣️ Try **QLoRA** (`quantization_bit: 4`) on a bigger model like Qwen2.5-7B
- 🌍 Fine-tune on **your own data** — just extend `hello_alpaca.json`
- 🎛️ Explore the **LLaMA-Factory WebUI**: `llamafactory-cli webui`
- ⚖️ Learn about **DPO/RLHF** stages for alignment

Found an issue? [Submit one here](https://github.com/AidinZe/Hello-word-of-fine-tuning/issues/new)!